In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [3]:
df = pd.read_csv('data/heart_disease_uci.csv')

In [4]:
# Step 1: Drop irrelevant column & create binary target
df_clean = df.copy()
df_clean.drop(columns=['id'], inplace=True)
df_clean['target'] = df_clean['num'].apply(lambda x: 0 if x == 0 else 1)
df_clean.drop(columns=['num'], inplace=True)

print(f'Shape after drop: {df_clean.shape}')
print(f'Target — 0 (No Disease): {(df_clean["target"]==0).sum()}, 1 (Disease): {(df_clean["target"]==1).sum()}')

# Step 2: Encode categoricals (no leakage risk for encoding structure)
ohe_cols = ['cp', 'restecg', 'slope', 'thal']
df_clean = pd.get_dummies(df_clean, columns=ohe_cols, drop_first=False)

df_clean['sex']     = df_clean['sex'].map({'Male': 1, 'Female': 0})
df_clean['dataset'] = df_clean['dataset'].astype('category').cat.codes

for col in ['fbs', 'exang']:
    df_clean[col] = df_clean[col].map({True: 1, False: 0, 'True': 1, 'False': 0})

print(f'Shape after encoding: {df_clean.shape}')

# Step 3: Train / Test split FIRST (stratified 80/20)
X = df_clean.drop(columns=['target'])
y = df_clean['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')
print(f'Train class balance: {y_train.value_counts().to_dict()}')
print(f'Test class balance:  {y_test.value_counts().to_dict()}')

# Step 4: Impute missing values (fit on TRAIN only, apply to both)
numeric_cols = ['trestbps', 'chol', 'thalch', 'oldpeak', 'ca']
bool_cols    = ['fbs', 'exang']

train_medians = X_train[numeric_cols].median()
train_modes   = X_train[bool_cols].mode().iloc[0]

X_train[numeric_cols] = X_train[numeric_cols].fillna(train_medians)
X_test[numeric_cols]  = X_test[numeric_cols].fillna(train_medians)
X_train[bool_cols]    = X_train[bool_cols].fillna(train_modes)
X_test[bool_cols]     = X_test[bool_cols].fillna(train_modes)

print(f'Remaining missing — Train: {X_train.isnull().sum().sum()}, Test: {X_test.isnull().sum().sum()}')

# Step 5: Outlier capping via IQR (fit on TRAIN only, apply to both)
outlier_cols = ['trestbps', 'chol', 'thalch', 'oldpeak']

for col in outlier_cols:
    Q1    = X_train[col].quantile(0.25)
    Q3    = X_train[col].quantile(0.75)
    IQR   = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    X_train[col] = X_train[col].clip(lower=lower, upper=upper)
    X_test[col]  = X_test[col].clip(lower=lower, upper=upper)
    print(f'  {col}: capped [{lower:.2f}, {upper:.2f}]')

# Step 6: Feature scaling (fit on TRAIN only, apply to both)
scale_cols = ['age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'ca']
scaler = StandardScaler()

X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test[scale_cols]  = scaler.transform(X_test[scale_cols])

print(f'Scaling applied. Final feature count: {X_train.shape[1]}')

# Step 7: Save processed data
X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv',   index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv',   index=False)

print('Saved: X_train.csv, X_test.csv, y_train.csv, y_test.csv')
print('Milestone 1 complete.')

Shape after drop: (920, 15)
Target — 0 (No Disease): 411, 1 (Disease): 509
Shape after encoding: (920, 24)
Train: 736 samples | Test: 184 samples
Train class balance: {1: 407, 0: 329}
Test class balance:  {1: 102, 0: 82}
Remaining missing — Train: 0, Test: 0
  trestbps: capped [90.00, 170.00]
  chol: capped [43.12, 400.12]
  thalch: capped [67.50, 207.50]
  oldpeak: capped [-2.25, 3.75]
Scaling applied. Final feature count: 23
Saved: X_train.csv, X_test.csv, y_train.csv, y_test.csv
Milestone 1 complete.
